# DataSage — Stage 3: Answering GRPO Training

Trains Qwen2.5-3B via **Unsloth** + **TRL GRPO** to generate persona-aware, data-grounded answers against the DataSage Answering OpenEnv environment deployed on HuggingFace Spaces.

Includes **Patronus Lynx** integration for hallucination detection as a reward signal.

**Environment:** [ricalanis/datasage-answering](https://huggingface.co/spaces/ricalanis/datasage-answering)  
**Model:** Qwen2.5-3B-Instruct (4-bit via Unsloth)  
**Framework:** TRL GRPOTrainer with environment-in-the-loop + Patronus Lynx rewards

In [ ]:
!pip install -q "openenv-core[core]>=0.2.1"
!pip install -q trl>=0.26 transformers accelerate peft bitsandbytes
!pip install -q vllm
!pip install -q unsloth
!pip install -q wandb patronus

In [ ]:
import json
import os
import re
import torch
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# === Configuration ===
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-answering.hf.space"
HF_MODEL_REPO = "ricalanis/datasage-qwen-answering"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["HF_TOKEN"] = "your_token_here"
# os.environ["WANDB_API_KEY"] = "your_key_here"
# os.environ["PATRONUS_API_KEY"] = "your_key_here"  # Optional: for Patronus Lynx

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")

In [ ]:
SYSTEM_PROMPT = """\
You are a data-driven enterprise analyst. You answer questions about \
datasets across multiple domains (HR, Sales, Project Management, \
IT Operations) tailored to the audience persona.

Personas:
- Executive: Focus on costs, ROI, strategic risk, portfolio trends, \
year-over-year metrics. Use strategic-financial language.
- Manager: Focus on team performance, operational health, process \
bottlenecks, capacity. Use operational-actionable language.
- Individual Contributor: Focus on personal tasks, deadlines, what to \
do next, simple explanations. Use plain-personal language.

Respond with a JSON answer:
{"answer": "<your answer>", "cited_columns": ["col1", "col2"], \
"reasoning": "<chain-of-thought>"}

Rules:
1. Ground every claim in the data \u2014 cite specific columns and statistics.
2. Match your tone and vocabulary to the persona.
3. Be concise but thorough \u2014 Executives want bullet points, ICs want \
clear next steps.
4. Never fabricate numbers \u2014 if the data doesn't support a claim, say so.

Think step by step: identify the persona, understand the question, \
examine the data, then answer."""

TASK_PROMPTS = [
    # HR domain
    "As an Executive: What is the overall attrition rate and its financial impact on the organization?",
    "As a Manager: Which departments have the highest turnover and what patterns do you see?",
    "As an Individual Contributor: Am I at risk of burnout based on the overtime data?",
    "As an Executive: How does our compensation compare to industry benchmarks across job roles?",
    "As a Manager: Which team members show the highest flight risk scores?",
    "As an Individual Contributor: What's the average tenure in my department and how do I compare?",
    "As an Executive: What's the year-over-year trend in employee satisfaction scores?",
    "As a Manager: How is overtime distributed across my team and is it sustainable?",
    # Sales domain
    "As an Executive: What's the pipeline health and forecast accuracy for this quarter?",
    "As a Manager: Which deals are at risk of slipping from the forecast?",
    "As an Individual Contributor: What should I focus on to close my highest-probability deal?",
    "As an Executive: What's the revenue impact of deals stuck in Negotiation stage?",
    "As a Manager: How does our team's deal velocity compare across regions?",
    "As an Individual Contributor: Which of my deals has the best win probability?",
    "As an Executive: What's the conversion rate by stage and where are we losing deals?",
    "As a Manager: Which reps are below quota and what's their pipeline gap?",
    # PM domain
    "As an Executive: Which projects are at highest risk of missing their deadlines?",
    "As a Manager: How is resource utilization distributed across the team?",
    "As an Individual Contributor: Which of my tasks should I prioritize based on dependencies?",
    "As an Executive: What's the portfolio-level on-time delivery rate?",
    "As a Manager: What's the burndown rate for the current sprint?",
    "As an Individual Contributor: Am I overallocated based on estimated vs actual hours?",
    "As an Executive: What's the financial impact of project delays in the portfolio?",
    "As a Manager: Which tasks are blocking the most downstream work?",
    # IT Ops domain
    "As an Executive: What's our SLA compliance rate and its trend this quarter?",
    "As a Manager: Which ticket categories have the worst resolution times?",
    "As an Individual Contributor: Which tickets assigned to me are closest to breaching SLA?",
    "As an Executive: What's the cost impact of SLA breaches this month?",
    "As a Manager: How should I prioritize the escalation queue?",
    "As an Individual Contributor: What's the resolution pattern for tickets like mine?",
    "As an Executive: What are the top 3 systemic issues driving incident volume?",
    "As a Manager: Which systems have the most recurring incidents?",
]


def make_conversation(user_msg):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]


dataset = Dataset.from_dict({
    "prompt": [make_conversation(p) for p in TASK_PROMPTS]
})

print(f"Dataset size: {len(dataset)} prompts across 4 domains x 3 personas")

## Reward Functions

Four reward signals drive GRPO training:
1. **Environment Reward** (primary): Steps through the OpenEnv answering environment
2. **Patronus Lynx Reward**: Hallucination detection via Patronus AI (with local fallback)
3. **JSON Format Reward**: Encourages well-structured JSON answer output
4. **Persona Reward**: Encourages persona-appropriate language and tone

In [ ]:
def parse_answering_action(text):
    """Parse model output into an answering action dict."""
    json_match = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "answer" in data:
                return data
        except json.JSONDecodeError:
            pass
    cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
    return {"answer": text, "cited_columns": cited[:5], "reasoning": ""}


def local_faithfulness_fn(completions, **kwargs):
    """Local faithfulness heuristic (fallback for Patronus)."""
    rewards = []
    for text in completions:
        score = 0.0
        lower = text.lower()
        cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
        if len(cited) >= 1:
            score += 0.3
        if len(cited) >= 3:
            score += 0.2
        if re.search(r'\d+\.?\d*%', text) or re.search(r'\b\d{2,}\b', text):
            score += 0.2
        if any(w in lower for w in ["i believe", "probably", "might be", "i'm not sure"]):
            score -= 0.2
        if len(text.strip()) < 50:
            score -= 0.3
        rewards.append(max(0.0, min(1.0, score)))
    return rewards


def patronus_reward_fn(completions, **kwargs):
    """Use Patronus Lynx for hallucination detection. Falls back to local scorer."""
    api_key = os.environ.get("PATRONUS_API_KEY")
    if not api_key:
        return local_faithfulness_fn(completions, **kwargs)
    try:
        import patronus
        patronus.init()
        from patronus import Patronus, RemoteEvaluator
        client = Patronus()
        lynx = RemoteEvaluator("lynx", "patronus:hallucination")
        context = kwargs.get("context", "")
        task_input = kwargs.get("task_input", "Answer the question based on the data.")
        rewards = []
        for text in completions:
            result = client.evaluate(
                evaluators=lynx,
                task_output=text,
                task_input=task_input,
                task_context=context,
            )
            rewards.append(float(result.results[0].score))
        return rewards
    except Exception:
        return local_faithfulness_fn(completions, **kwargs)


def env_reward_fn(completions, **kwargs):
    """Step through the Answering environment for each completion. Primary reward."""
    import requests
    rewards = []
    for text in completions:
        try:
            action_dict = parse_answering_action(text)
            requests.post(f"{ENV_URL}/reset", json={})
            step_resp = requests.post(f"{ENV_URL}/step", json={"action": action_dict})
            step_data = step_resp.json()
            rewards.append(float(step_data.get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards


def json_format_reward(completions, **kwargs):
    """Reward well-formed JSON answer actions."""
    rewards = []
    for text in completions:
        if re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL):
            try:
                match = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
                data = json.loads(match.group())
                has_answer = bool(data.get("answer", "").strip())
                has_cited = bool(data.get("cited_columns"))
                has_reasoning = bool(data.get("reasoning", "").strip())
                if has_answer and has_cited and has_reasoning:
                    rewards.append(1.0)
                elif has_answer and has_cited:
                    rewards.append(0.7)
                elif has_answer:
                    rewards.append(0.4)
                else:
                    rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


def persona_reward(completions, **kwargs):
    """Reward persona-appropriate language."""
    rewards = []
    for text in completions:
        score = 0.0
        lower = text.lower()
        exec_markers = ["roi", "revenue", "cost", "strategic", "quarter",
                        "year-over-year", "portfolio", "margin", "growth", "benchmark", "trend"]
        mgr_markers = ["team", "performance", "bottleneck", "capacity",
                       "action", "priority", "escalation", "delivery", "utilization"]
        ic_markers = ["my", "i should", "next step", "deadline", "help",
                      "understand", "focus on", "assigned to me"]
        exec_hits = sum(1 for m in exec_markers if m in lower)
        mgr_hits = sum(1 for m in mgr_markers if m in lower)
        ic_hits = sum(1 for m in ic_markers if m in lower)
        max_hits = max(exec_hits, mgr_hits, ic_hits)
        if max_hits >= 4:
            score = 0.5
        elif max_hits >= 2:
            score = 0.3
        elif max_hits >= 1:
            score = 0.1
        rewards.append(score)
    return rewards

## GRPO Training

Using Group Relative Policy Optimization with environment-in-the-loop + Patronus Lynx rewards. Config tuned for Colab T4 GPU (use larger batches on H100).

In [ ]:
training_args = GRPOConfig(
    output_dir="./outputs/answering-grpo",
    learning_rate=3e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=512,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb",
    run_name="datasage-answering-grpo",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        env_reward_fn,         # Primary: environment reward
        patronus_reward_fn,    # Faithfulness: Patronus Lynx (with local fallback)
        json_format_reward,    # Auxiliary: structured output
        persona_reward,        # Auxiliary: persona alignment
    ],
)

print("Starting Stage 3 (Answering) GRPO training...")
trainer.train()

In [ ]:
output_dir = "./outputs/answering-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Training complete! Model saved to {output_dir}")

trainer.push_to_hub(HF_MODEL_REPO)
print(f"Model pushed to https://huggingface.co/{HF_MODEL_REPO}")